# Project Manager Agent - Evaluation Experiments

This notebook provides experimental evaluation of the `AgenticProjectManager` agent.

## Experiments:
1. **Table 4.4: Intent Classification Accuracy (Router Node)** - Evaluates the router's ability to correctly classify user intents.
2. **Agent Reasoning Trace for Complex Request (Tool Call)** - Evaluates tool-calling reasoning with reference comparison metrics.

**Pre-requisites:**
- `.env` file in project root with `DATABASE_URL` and `GEMINI_API_KEY`
- Backend API running (for tool execution tests)

In [3]:
import sys
import os
from dotenv import load_dotenv

# Setup paths
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..', '..'))
ai_service_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.append(project_root)
if ai_service_root not in sys.path:
    sys.path.append(ai_service_root)

# Load environment variables
dotenv_path = os.path.join(project_root, '.env')
load_dotenv(dotenv_path)

print(f"Project Root: {project_root}")
print(f"DATABASE_URL set: {bool(os.getenv('DATABASE_URL'))}")
print(f"GEMINI_API_KEY set: {bool(os.getenv('GEMINI_API_KEY'))}")

Project Root: e:\Individual\Projects\ProMeet
DATABASE_URL set: True
GEMINI_API_KEY set: True


In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from collections import Counter
import time

# Import agent
from src.agents.project_manager.agent import AgenticProjectManager, AgentState, RouterOutput
from langchain_core.messages import SystemMessage, HumanMessage

print("All imports successful!")

All imports successful!


---
# Experiment 1: Intent Classification Accuracy (Router Node)

**Table 4.4: Intent Classification Accuracy**

Evaluates the Router node's ability to classify user queries into:
- `DIRECT`: Small talk, general knowledge, writing assistance
- `TOOL_CALL`: Operations requiring API/tool access

### Metrics:
- Accuracy
- Precision (per class)
- Recall (per class)
- F1-Score

In [5]:
# ==============================================================================
# INTENT CLASSIFICATION DATASET
# Define your test queries and expected labels here
# ==============================================================================

intent_test_cases = [
    # --- DIRECT Intent Examples ---
    {"query": "Xin chào, bạn có khỏe không?", "expected": "DIRECT", "category": "greeting"},
    {"query": "Cảm ơn bạn rất nhiều!", "expected": "DIRECT", "category": "thanks"},
    {"query": "Scrum là gì?", "expected": "DIRECT", "category": "knowledge"},
    {"query": "Viết email xin hoãn deadline cho khách hàng", "expected": "DIRECT", "category": "writing"},
    {"query": "REST API hoạt động như thế nào?", "expected": "DIRECT", "category": "knowledge"},
    
    # --- TOOL_CALL Intent Examples ---
    {"query": "Danh sách dự án của tôi", "expected": "TOOL_CALL", "category": "list_projects"},
    {"query": "Các task trong dự án AI App", "expected": "TOOL_CALL", "category": "list_tasks"},
    {"query": "Tạo task mới: Review code", "expected": "TOOL_CALL", "category": "create_task"},
    {"query": "Cập nhật trạng thái task #5 sang Done", "expected": "TOOL_CALL", "category": "update_task"},
    {"query": "Thông tin tài khoản của tôi", "expected": "TOOL_CALL", "category": "user_info"},
    
    # --- ADD MORE TEST CASES BELOW ---
    # {"query": "Your query here", "expected": "DIRECT" or "TOOL_CALL", "category": "category_name"},
]

print(f"Total test cases: {len(intent_test_cases)}")
print(f"DIRECT cases: {len([t for t in intent_test_cases if t['expected'] == 'DIRECT'])}")
print(f"TOOL_CALL cases: {len([t for t in intent_test_cases if t['expected'] == 'TOOL_CALL'])}")

Total test cases: 10
DIRECT cases: 5
TOOL_CALL cases: 5


In [6]:
# Initialize agent
agent = AgenticProjectManager()
print("Agent initialized successfully.")

Agent initialized successfully.


In [7]:
def evaluate_router(agent, test_cases: List[Dict]) -> pd.DataFrame:
    """
    Evaluate the router node on intent classification.
    
    Returns:
        DataFrame with results for each test case
    """
    results = []
    
    for i, case in enumerate(test_cases):
        query = case["query"]
        expected = case["expected"]
        category = case.get("category", "unknown")
        
        # Create state and call router
        state = {
            "messages": [],
            "query": query,
            "router_decision": ""
        }
        
        start_time = time.time()
        try:
            router_output = agent.router(state)
            predicted = router_output.get("router_decision", "ERROR")
        except Exception as e:
            predicted = "ERROR"
            print(f"Error on query '{query}': {e}")
        latency = time.time() - start_time
        
        correct = predicted == expected
        
        results.append({
            "index": i + 1,
            "query": query,
            "category": category,
            "expected": expected,
            "predicted": predicted,
            "correct": correct,
            "latency_s": round(latency, 3)
        })
        
        status = "✓" if correct else "✗"
        print(f"[{i+1}/{len(test_cases)}] {status} Query: '{query[:40]}...' | Expected: {expected} | Predicted: {predicted}")
    
    return pd.DataFrame(results)

# Run evaluation
print("\n" + "="*80)
print("Running Intent Classification Evaluation...")
print("="*80 + "\n")

results_df = evaluate_router(agent, intent_test_cases)


Running Intent Classification Evaluation...

[1/10] ✓ Query: 'Xin chào, bạn có khỏe không?...' | Expected: DIRECT | Predicted: DIRECT
[2/10] ✓ Query: 'Cảm ơn bạn rất nhiều!...' | Expected: DIRECT | Predicted: DIRECT
[3/10] ✓ Query: 'Scrum là gì?...' | Expected: DIRECT | Predicted: DIRECT
[4/10] ✓ Query: 'Viết email xin hoãn deadline cho khách h...' | Expected: DIRECT | Predicted: DIRECT


Router response is None or empty


[5/10] ✓ Query: 'REST API hoạt động như thế nào?...' | Expected: DIRECT | Predicted: DIRECT
[6/10] ✓ Query: 'Danh sách dự án của tôi...' | Expected: TOOL_CALL | Predicted: TOOL_CALL
[7/10] ✓ Query: 'Các task trong dự án AI App...' | Expected: TOOL_CALL | Predicted: TOOL_CALL
[8/10] ✓ Query: 'Tạo task mới: Review code...' | Expected: TOOL_CALL | Predicted: TOOL_CALL
[9/10] ✓ Query: 'Cập nhật trạng thái task #5 sang Done...' | Expected: TOOL_CALL | Predicted: TOOL_CALL
[10/10] ✓ Query: 'Thông tin tài khoản của tôi...' | Expected: TOOL_CALL | Predicted: TOOL_CALL


In [8]:
def compute_classification_metrics(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Compute classification metrics from results DataFrame.
    """
    y_true = df["expected"].tolist()
    y_pred = df["predicted"].tolist()
    
    # Filter out errors for metric calculation
    valid_mask = df["predicted"] != "ERROR"
    y_true_valid = df[valid_mask]["expected"].tolist()
    y_pred_valid = df[valid_mask]["predicted"].tolist()
    
    labels = ["DIRECT", "TOOL_CALL"]
    
    metrics = {
        "total_samples": len(df),
        "valid_samples": len(y_true_valid),
        "error_samples": len(df) - len(y_true_valid),
        "accuracy": accuracy_score(y_true_valid, y_pred_valid) if y_true_valid else 0,
        "precision_macro": precision_score(y_true_valid, y_pred_valid, labels=labels, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_valid, y_pred_valid, labels=labels, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true_valid, y_pred_valid, labels=labels, average="macro", zero_division=0),
        "avg_latency_s": df["latency_s"].mean(),
    }
    
    # Per-class metrics
    for label in labels:
        metrics[f"precision_{label}"] = precision_score(y_true_valid, y_pred_valid, labels=[label], average="micro", zero_division=0)
        metrics[f"recall_{label}"] = recall_score(y_true_valid, y_pred_valid, labels=[label], average="micro", zero_division=0)
        metrics[f"f1_{label}"] = f1_score(y_true_valid, y_pred_valid, labels=[label], average="micro", zero_division=0)
    
    return metrics

# Compute and display metrics
metrics = compute_classification_metrics(results_df)

print("\n" + "="*80)
print("TABLE 4.4: INTENT CLASSIFICATION ACCURACY (ROUTER NODE)")
print("="*80)
print(f"\n{'Metric':<30} {'Value':>15}")
print("-"*45)
print(f"{'Total Samples':<30} {metrics['total_samples']:>15}")
print(f"{'Valid Samples':<30} {metrics['valid_samples']:>15}")
print(f"{'Error Samples':<30} {metrics['error_samples']:>15}")
print("-"*45)
print(f"{'Accuracy':<30} {metrics['accuracy']:>15.4f}")
print(f"{'Precision (Macro)':<30} {metrics['precision_macro']:>15.4f}")
print(f"{'Recall (Macro)':<30} {metrics['recall_macro']:>15.4f}")
print(f"{'F1-Score (Macro)':<30} {metrics['f1_macro']:>15.4f}")
print("-"*45)
print(f"{'Avg Latency (s)':<30} {metrics['avg_latency_s']:>15.3f}")
print("="*80)


TABLE 4.4: INTENT CLASSIFICATION ACCURACY (ROUTER NODE)

Metric                                   Value
---------------------------------------------
Total Samples                               10
Valid Samples                               10
Error Samples                                0
---------------------------------------------
Accuracy                                1.0000
Precision (Macro)                       1.0000
Recall (Macro)                          1.0000
F1-Score (Macro)                        1.0000
---------------------------------------------
Avg Latency (s)                          0.875


In [9]:
# Display detailed results and confusion matrix
from sklearn.metrics import classification_report

valid_mask = results_df["predicted"] != "ERROR"
y_true = results_df[valid_mask]["expected"].tolist()
y_pred = results_df[valid_mask]["predicted"].tolist()

print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_true, y_pred, target_names=["DIRECT", "TOOL_CALL"]))

print("\nCONFUSION MATRIX")
print("-"*40)
cm = confusion_matrix(y_true, y_pred, labels=["DIRECT", "TOOL_CALL"])
cm_df = pd.DataFrame(cm, index=["Actual: DIRECT", "Actual: TOOL_CALL"], 
                      columns=["Pred: DIRECT", "Pred: TOOL_CALL"])
print(cm_df)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

      DIRECT       1.00      1.00      1.00         5
   TOOL_CALL       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


CONFUSION MATRIX
----------------------------------------
                   Pred: DIRECT  Pred: TOOL_CALL
Actual: DIRECT                5                0
Actual: TOOL_CALL             0                5


In [10]:
# Display full results table
print("\n" + "="*80)
print("DETAILED RESULTS")
print("="*80)
display(results_df)


DETAILED RESULTS


,index,query,category,expected,predicted,correct,latency_s
0,1,"Xin chào, bạn có khỏe không?",greeting,DIRECT,DIRECT,True,1.020
1,2,Cảm ơn bạn rất nhiều!,thanks,DIRECT,DIRECT,True,0.765
2,3,Scrum là gì?,knowledge,DIRECT,DIRECT,True,0.847
3,4,Viết email xin hoãn deadline cho khách hàng,writing,DIRECT,DIRECT,True,0.838
4,5,REST API hoạt động như thế nào?,knowledge,DIRECT,DIRECT,True,1.319
5,6,Danh sách dự án của tôi,list_projects,TOOL_CALL,TOOL_CALL,True,0.740
6,7,Các task trong dự án AI App,list_tasks,TOOL_CALL,TOOL_CALL,True,0.744
7,8,Tạo task mới: Review code,create_task,TOOL_CALL,TOOL_CALL,True,0.816
8,9,Cập nhật trạng thái task #5 sang Done,update_task,TOOL_CALL,TOOL_CALL,True,0.751
9,10,Thông tin tài khoản của tôi,user_info,TOOL_CALL,TOOL_CALL,True,0.913
